# A verified token becomes a user with role classes

The HTTP coordinator verifies a token and maps the manager role name to ManagerRole.

Run with the Python environment of this checkout (`.venv`). These examples use the API in this branch; installing a released package may give a different API.

[Script](12_authentication.py) · [Tutorial](../../docs/tutorials/step-03-authorization-and-roles.md)

## Imports

In [1]:
"""A verified token becomes a user with role classes."""

from __future__ import annotations

import asyncio
import time

import jwt
from starlette.requests import Request

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.auth.jwt_auth import JwtAuthCoordinator

## The rule

The HTTP coordinator verifies a token and maps the manager role name to ManagerRole.

In [2]:
class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


# This key is only for this local example.
DEMO_KEY = "local-tutorial-key-not-for-production-123456"

## Try it

Run the cells in order. The calls below are the same as in the script.

In [3]:
auth = JwtAuthCoordinator(secret_key=DEMO_KEY, role_registry={"manager": ManagerRole})
token = jwt.encode({"sub": "m-1", "roles": ["manager"], "exp": int(time.time()) + 60}, DEMO_KEY, algorithm="HS256")
request = Request(
    {
        "type": "http",
        "method": "POST",
        "path": "/orders/cancel",
        "headers": [(b"authorization", f"Bearer {token}".encode())],
        "query_string": b"",
        "server": ("localhost", 80),
        "scheme": "http",
    }
)
context = await auth.process(request)
print("user:", context.user.user_id)
print("roles:", [role.__name__ for role in context.user.roles])

user: m-1
roles: ['ManagerRole']


In [4]:
invalid = Request({**request.scope, "headers": [(b"authorization", b"Bearer invalid-token")]})
print("invalid token:", await auth.process(invalid))

invalid token: None


## What the result means

The valid token produces a Context with a typed role. Invalid credentials produce None; the adapter rejects the request before machine.run. Manually constructing Context in the other examples bypasses this authentication step.